In [1]:
import pandas as pd

df = pd.read_csv("../raw/1522_modalità_conoscenza.csv")

df.head()

,FREQ,Frequenza,REF_AREA,Territorio,DATA_TYPE,Indicatore,NATIONALITY,Nazionalità,KNOWLEDGE_1522,Modalità di conoscenza del 1522,TIME_PERIOD,Osservazione,OBS_STATUS
0,A,Annuale,IT,Italia,USERS,Utenti del 1522,WORLD,Mondo,BROCHURE,Manifesti e locandine,2015,103,NaN
1,A,Annuale,IT,Italia,USERS,Utenti del 1522,WORLD,Mondo,BROCHURE,Manifesti e locandine,2016,112,NaN
2,A,Annuale,IT,Italia,USERS,Utenti del 1522,WORLD,Mondo,BROCHURE,Manifesti e locandine,2017,96,NaN
3,A,Annuale,IT,Italia,USERS,Utenti del 1522,WORLD,Mondo,BROCHURE,Manifesti e locandine,2018,71,NaN
4,A,Annuale,IT,Italia,USERS,Utenti del 1522,WORLD,Mondo,BROCHURE,Manifesti e locandine,2019,84,NaN


In [2]:
# Per ogni colonna: numero di valori unici + elenco valori

for col in df.columns:
    print("=" * 60)
    print(f"Colonna: {col}")
    print(f"Numero valori unici: {df[col].nunique(dropna=False)}")
    print("Valori unici:")
    print(df[col].unique())
    print("\n")

Colonna: FREQ
Numero valori unici: 1
Valori unici:
['A']


Colonna: Frequenza
Numero valori unici: 1
Valori unici:
['Annuale']


Colonna: REF_AREA
Numero valori unici: 27
Valori unici:
['IT' 'ITC' 'ITC1' 'ITC2' 'ITC3' 'ITC4' 'ITD' 'ITDA' 'ITD3' 'ITD4' 'ITD5'
 'ITE' 'ITE1' 'ITE2' 'ITE3' 'ITE4' 'ITF' 'ITF1' 'ITF2' 'ITF3' 'ITF4'
 'ITF5' 'ITF6' 'ITG' 'ITG1' 'ITG2' 'ITNI']


Colonna: Territorio
Numero valori unici: 27
Valori unici:
['Italia' 'Nord-ovest' 'Piemonte' '\'Valle d"\'Aosta / Vallée d"\'Aoste\''
 'Liguria' 'Lombardia' 'Nord-est' 'Trentino Alto Adige / Südtirol'
 'Veneto' 'Friuli-Venezia Giulia' 'Emilia-Romagna' 'Centro' 'Toscana'
 'Umbria' 'Marche' 'Lazio' 'Sud' 'Abruzzo' 'Molise' 'Campania' 'Puglia'
 'Basilicata' 'Calabria' 'Isole' 'Sicilia' 'Sardegna' 'Non indicato']


Colonna: DATA_TYPE
Numero valori unici: 1
Valori unici:
['USERS']


Colonna: Indicatore
Numero valori unici: 1
Valori unici:
['Utenti del 1522']


Colonna: NATIONALITY
Numero valori unici: 4
Valori unici:
['WORLD'

In [3]:
# Rimozione colonne non informative o ridondanti:
# - colonne costanti (un solo valore per tutto il dataset)
# - colonne duplicate sul piano informativo
# - colonne completamente vuote

cols_to_drop = [
    "FREQ",               
    "Frequenza",          
    "REF_AREA",          
    "DATA_TYPE",          
    "Indicatore", 
    "NATIONALITY",   
    "KNOWLEDGE_1522",      
    "OBS_STATUS" 
]

df = df.drop(columns=cols_to_drop)

# Verifica struttura finale
print("Nuova shape:", df.shape)
print("Colonne rimanenti:", df.columns.tolist())


Nuova shape: (10038, 5)
Colonne rimanenti: ['Territorio', 'Nazionalità', 'Modalità di conoscenza del 1522', 'TIME_PERIOD', 'Osservazione']


In [4]:
# Standardizzazione nomi colonne:
# - tutto minuscolo
# - nomi più chiari e coerenti

df = df.rename(columns={
    "Territorio": "territorio",
    "Nazionalità": "nazionalità_utente",
    "Modalità di conoscenza del 1522": "modalità_conoscenza_1522",
    "TIME_PERIOD": "periodo",
    "Osservazione": "numero_vittime"
})


# Eliminazione delle righe completamente vuote 
df = df.dropna(how="all")

# Stampa delle colonne e dei valori unici che contengono
for col in df.columns:
    print("\nColonna:", col)
    print(df[col].unique())


Colonna: territorio
['Italia' 'Nord-ovest' 'Piemonte' '\'Valle d"\'Aosta / Vallée d"\'Aoste\''
 'Liguria' 'Lombardia' 'Nord-est' 'Trentino Alto Adige / Südtirol'
 'Veneto' 'Friuli-Venezia Giulia' 'Emilia-Romagna' 'Centro' 'Toscana'
 'Umbria' 'Marche' 'Lazio' 'Sud' 'Abruzzo' 'Molise' 'Campania' 'Puglia'
 'Basilicata' 'Calabria' 'Isole' 'Sicilia' 'Sardegna' 'Non indicato']

Colonna: nazionalità_utente
['Mondo' 'Paesi esteri' 'Italia' 'Non Indicato']

Colonna: modalità_conoscenza_1522
['Manifesti e locandine' 'Radio' 'TV' 'Stampa' 'Internet' 'Social media'
 'Notizie dal web - ricerca sui motori di ricerca'
 'Chat di amici e conoscenti' 'Parente / amico / conoscente' 'Parente'
 'Amico/conoscente' 'Servizio pubblico' 'Elenco telefonico'
 'Strutture sanitarie' 'Consultorio' '\'FFOO - forze dell"\'ordine\''
 'Servizi sociali' 'Medico di famiglia' 'Luogo di culto' 'Scuola'
 'Psicologo/a psichiatra' 'Altro' 'Non sa/non ricorda' 'Non risponde'
 'Non disponibile' 'Totale']

Colonna: periodo
[201

In [5]:
# Pulizia e normalizzazione valori:
# - territorio
# - nazionalità_utente
# - modalità_conoscenza_1522
# - conversione tipi
# - aggregazione corretta dopo il raggruppamento


# 1) territorio: minuscolo + pulizia base
df["territorio"] = df["territorio"].str.lower().str.strip()

df["territorio"] = df["territorio"].str.replace("-", " ", regex=False)

df["territorio"] = df["territorio"].replace({
    "'valle d\"'aosta / vallée d\"'aoste'": "valle d'aosta",
    "trentino alto adige / südtirol": "trentino alto adige"
})


# 2) nazionalità_utente: minuscolo + accorpamento categorie
df["nazionalità_utente"] = df["nazionalità_utente"].str.lower().str.strip()

df["nazionalità_utente"] = df["nazionalità_utente"].replace({
    "mondo": "estero",
    "paesi esteri": "estero",
    "non indicato": "non indicato"
})


# 3) modalità_conoscenza_1522: pulizia caratteri sporchi
df["modalità_conoscenza_1522"] = df["modalità_conoscenza_1522"].str.lower().str.strip()

df["modalità_conoscenza_1522"] = df["modalità_conoscenza_1522"].str.replace(
    '\'ffoo - forze dell"\'ordine\'',
    "forze dell'ordine",
    regex=False
)


# 4) rimozione "totale" prima del raggruppamento
df = df[df["modalità_conoscenza_1522"] != "totale"].copy()


# 5) raggruppamento in macro-categorie
modalita_map = {
    "manifesti e locandine": "media_tradizionali",
    "radio": "media_tradizionali",
    "tv": "media_tradizionali",
    "stampa": "media_tradizionali",
    "internet": "digitale",
    "social media": "digitale",
    "notizie dal web - ricerca sui motori di ricerca": "digitale",
    "chat di amici e conoscenti": "rete_personale",
    "parente / amico / conoscente": "rete_personale",
    "parente": "rete_personale",
    "amico/conoscente": "rete_personale",
    "servizio pubblico": "servizi_istituzioni",
    "elenco telefonico": "servizi_istituzioni",
    "strutture sanitarie": "servizi_istituzioni",
    "consultorio": "servizi_istituzioni",
    "forze dell'ordine": "servizi_istituzioni",
    "servizi sociali": "servizi_istituzioni",
    "medico di famiglia": "servizi_istituzioni",
    "luogo di culto": "servizi_istituzioni",
    "scuola": "servizi_istituzioni",
    "psicologo/a psichiatra": "servizi_istituzioni",
    "altro": "altro",
    "non sa/non ricorda": "non_risposta",
    "non risponde": "non_risposta",
    "non disponibile": "non_risposta"
}

df["modalità_conoscenza_1522"] = df["modalità_conoscenza_1522"].replace(modalita_map)

macro_categorie = {
    "media_tradizionali",
    "digitale",
    "rete_personale",
    "servizi_istituzioni",
    "altro",
    "non_risposta"
}

mask_altro = ~df["modalità_conoscenza_1522"].isin(macro_categorie)

df.loc[mask_altro, "modalità_conoscenza_1522"] = "altro"


# 6) conversione colonne numeriche
df["periodo"] = df["periodo"].astype(int)

df["numero_vittime"] = df["numero_vittime"].astype(int)


# 7) aggregazione corretta
df = (
    df.groupby(
        ["territorio", "nazionalità_utente", "modalità_conoscenza_1522", "periodo"],
        as_index=False
    )["numero_vittime"]
    .sum()
)


# Verifica finale post-pulizia (struttura + qualità base)

# 1) Dimensioni e info colonne
display(df.head(10))
df.info()

# 2) Tipi e missing
print("\nMissing values per colonna:")
display(df.isna().sum().sort_values(ascending=False))

# 3) Valori unici (conteggio) per colonna
print("\nNumero valori unici per colonna:")
display(df.nunique(dropna=False).sort_values(ascending=False))

# 4) Check duplicati (righe identiche)
print("\nRighe duplicate (identiche su tutte le colonne):", df.duplicated().sum())

# 5) Valori distinti per ogni colonna
for col in df.columns:
    print("\nColonna:", col)
    
    if col == "periodo":
        print(sorted(df[col].astype(int).unique().tolist()))
    else:
        print(df[col].unique())

,territorio,nazionalità_utente,modalità_conoscenza_1522,periodo,numero_vittime
0,abruzzo,estero,altro,2015,1
1,abruzzo,estero,altro,2016,4
2,abruzzo,estero,altro,2017,2
3,abruzzo,estero,altro,2018,2
4,abruzzo,estero,altro,2019,6
5,abruzzo,estero,altro,2020,8
6,abruzzo,estero,altro,2021,8
7,abruzzo,estero,altro,2022,6
8,abruzzo,estero,altro,2023,18
9,abruzzo,estero,altro,2024,6


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3195 entries, 0 to 3194
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   territorio                3195 non-null   object
 1   nazionalità_utente        3195 non-null   object
 2   modalità_conoscenza_1522  3195 non-null   object
 3   periodo                   3195 non-null   int64 
 4   numero_vittime            3195 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 124.9+ KB

Missing values per colonna:


territorio                  0
nazionalità_utente          0
modalità_conoscenza_1522    0
periodo                     0
numero_vittime              0
dtype: int64


Numero valori unici per colonna:


numero_vittime              993
territorio                   27
periodo                      10
modalità_conoscenza_1522      6
nazionalità_utente            3
dtype: int64


Righe duplicate (identiche su tutte le colonne): 0

Colonna: territorio
['abruzzo' 'basilicata' 'calabria' 'campania' 'centro' 'emilia romagna'
 'friuli venezia giulia' 'isole' 'italia' 'lazio' 'liguria' 'lombardia'
 'marche' 'molise' 'non indicato' 'nord est' 'nord ovest' 'piemonte'
 'puglia' 'sardegna' 'sicilia' 'sud' 'toscana' 'trentino alto adige'
 'umbria' "valle d'aosta" 'veneto']

Colonna: nazionalità_utente
['estero' 'italia' 'non indicato']

Colonna: modalità_conoscenza_1522
['altro' 'digitale' 'media_tradizionali' 'non_risposta' 'rete_personale'
 'servizi_istituzioni']

Colonna: periodo
[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Colonna: numero_vittime
[    1     4     2     6     8    18    51    59    74    89   145    96
   111   147   216   160   129    86    64   109    63   112    75    71
   146   410   317   277   234   327   284   364   276   337   214    19
    24    29    20    26    46    34    32    17    35    39    43    30
    21    62    4

In [6]:
# Salvataggio dataset pulito

output_path = "../data_clean/1522_modalità_conoscenza.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("File salvato in:", output_path)

File salvato in: ../data_clean/1522_modalità_conoscenza.csv
